Contrastive SSL: SimCLR, MoCo, CLIP, and SigLIP

Contrastive self-supervised and multimodal representation learning:
1. SimCLR: Normalized temperature-scaled cross-entropy (NT-Xent) over augmented pairs.
2. MoCo: Momentum contrast key encoder with asynchronous negative memory queues.
3. CLIP: Symmetric dual-encoder cross-entropy for image-text multimodal alignment.
4. SigLIP: Pairwise binary sigmoid contrastive loss eliminating all-gather bottlenecks.

## 2. Mathematical Formulations & Theoretical Foundations

- **SimCLR NT-Xent Loss**:
  $$\mathcal{L}_{\text{NT-Xent}} = -\log \frac{\exp(\langle \hat{z}_1, \hat{z}_2 \rangle / \tau)}{\sum_k \exp(\langle \hat{z}_1, \hat{z}_k \rangle / \tau)}$$
- **MoCo Memory Queue**:
  $$\mathcal{L}_{\text{MoCo}} = -\log \frac{\exp(q \cdot k^+ / \tau)}{\exp(q \cdot k^+ / \tau) + \sum_{m} \exp(q \cdot k_m^- / \tau)}$$
- **CLIP Symmetric Cross-Entropy**:
  $$\mathcal{L}_{\text{CLIP}} = \frac{1}{2} \left( \text{CE}(L, \mathbf{y}) + \text{CE}(L^T, \mathbf{y}) \right)$$
- **SigLIP Pairwise Loss**:
  $$\mathcal{L}_{\text{SigLIP}} = \frac{1}{B} \sum_{i, j} \log\left(1 + \exp(-Y_{ij} L_{ij})\right)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "08_learning_paradigms_rl_and_alignment" / "01_self_semi_meta_continual_federated" / "01_contrastive_ssl_simclr_moco_and_clip_siglip.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. SimCLR NT-Xent Loss
def simclr_nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    z1_norm = F.normalize(z1, dim=-1)
    z2_norm = F.normalize(z2, dim=-1)
    sim_matrix = (z1_norm @ z2_norm.t()) / temperature
    labels = torch.arange(z1.shape[0], device=z1.device)
    return F.cross_entropy(sim_matrix, labels)

simclr_calc = simclr_nt_xent_loss(golden["z1"], golden["z2"], temperature=0.1)
diff_simclr = torch.abs(simclr_calc - golden["simclr_ref"]).item()
print(f"SimCLR NT-Xent Loss max abs error: {diff_simclr:.2e}")
assert diff_simclr < 1e-5

In [ ]:
# 2. MoCo Contrastive Loss
def moco_contrastive_loss(
    q: torch.Tensor,
    k_pos: torch.Tensor,
    queue: torch.Tensor,
    temperature: float = 0.07,
) -> torch.Tensor:
    q_norm = F.normalize(q, dim=-1)
    k_norm = F.normalize(k_pos, dim=-1)
    queue_norm = F.normalize(queue, dim=-1)
    l_pos = torch.sum(q_norm * k_norm, dim=-1, keepdim=True) / temperature
    l_neg = (q_norm @ queue_norm.t()) / temperature
    logits = torch.cat([l_pos, l_neg], dim=1)
    labels = torch.zeros(q.shape[0], dtype=torch.long, device=q.device)
    return F.cross_entropy(logits, labels)

moco_calc = moco_contrastive_loss(
    golden["moco_q"], golden["moco_k_pos"], golden["moco_queue"], temperature=0.1
)
diff_moco = torch.abs(moco_calc - golden["moco_ref"]).item()
print(f"MoCo Contrastive Loss max abs error: {diff_moco:.2e}")
assert diff_moco < 1e-5

In [ ]:
# 3. CLIP Symmetric Loss
def clip_symmetric_loss(
    image_embeds: torch.Tensor,
    text_embeds: torch.Tensor,
    log_temp: torch.Tensor,
    bias: torch.Tensor,
) -> torch.Tensor:
    i_norm = F.normalize(image_embeds, dim=-1)
    t_norm = F.normalize(text_embeds, dim=-1)
    temp = torch.exp(torch.clamp(log_temp, max=4.6052))
    logits = (i_norm @ t_norm.t()) * temp + bias
    labels = torch.arange(image_embeds.shape[0], device=image_embeds.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss_i2t + loss_t2i)

clip_calc = clip_symmetric_loss(
    golden["img_embeds"], golden["txt_embeds"], golden["log_temp"], golden["clip_bias"]
)
diff_clip = torch.abs(clip_calc - golden["clip_ref"]).item()
print(f"CLIP Symmetric Loss max abs error: {diff_clip:.2e}")
assert diff_clip < 1e-5

In [ ]:
# 4. SigLIP Pairwise Loss & Summary Table
def siglip_pairwise_loss(
    image_embeds: torch.Tensor,
    text_embeds: torch.Tensor,
    log_temp: torch.Tensor,
    bias: torch.Tensor,
) -> torch.Tensor:
    b = image_embeds.shape[0]
    i_norm = F.normalize(image_embeds, dim=-1)
    t_norm = F.normalize(text_embeds, dim=-1)
    temp = torch.exp(torch.clamp(log_temp, max=4.6052))
    logits = (i_norm @ t_norm.t()) * temp + bias
    labels = 2.0 * torch.eye(b, device=image_embeds.device) - 1.0
    loss = F.softplus(-labels * logits)
    return loss.mean()

siglip_calc = siglip_pairwise_loss(
    golden["img_embeds"], golden["txt_embeds"], golden["log_temp"], golden["clip_bias"]
)
diff_siglip = torch.abs(siglip_calc - golden["siglip_ref"]).item()

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'SimCLR NT-Xent Loss':<32} | {diff_simclr:<15.2e} | PASSED")
print(f"{'MoCo Contrastive Loss':<32} | {diff_moco:<15.2e} | PASSED")
print(f"{'CLIP Symmetric Loss':<32} | {diff_clip:<15.2e} | PASSED")
print(f"{'SigLIP Pairwise Loss':<32} | {diff_siglip:<15.2e} | PASSED")
print("=" * 65)
assert diff_siglip < 1e-5

## 8. Summary, Invariants & Diagnostic Checklist

- [x] SimCLR leverages temperature-scaled normalized similarities over positive/negative views.
- [x] MoCo decouples dictionary size from GPU mini-batch capacity via slow momentum key encoder updates.
- [x] CLIP aligns multimodal representation spaces through symmetric bidirectional cross-entropy.
- [x] SigLIP replaces all-gather softmax with decoupled pairwise binary sigmoid losses, accelerating distributed training.